![image.png](https://i.imgur.com/a3uAqnb.png)
# Lab 6: Large Reasoning Models (GRPO / GSPO)

This notebook focuses on **reasoning-oriented LLM training** — chain-of-thought prompting, rule-based verifiers, and **GRPO** group advantages.

You will implement GRPO normalization, a GSM8K-style answer extractor, and run inference on a small math-oriented language model.

> 💡 GRPO replaces a critic with **group-relative** advantages — a key idea from DeepSeek-R1-style post-training.

__Let's install libraries and implement the building blocks.__



# 📦 Installing Required Python Libraries

This cell installs packages needed for this lab.

- **PyTorch** — Tensor ops for GRPO advantage computation.
- **Transformers / Accelerate** — Load and run instruction-tuned LLMs.
- **Bitsandbytes** (optional) — 4-bit loading for larger models on Colab GPU.


In [1]:
!pip install -q torch transformers accelerate bitsandbytes matplotlib numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.6 MB/s eta 0:00:00


# 📥 Importing Essential Python Libraries

Imports for GRPO utilities and LLM inference in Part B.


In [2]:
import re
import numpy as np
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


PyTorch 2.11.0+cpu
CUDA available: False


__Theory first — connect definitions to the lecture slides, then move to code.__

---

## 🧠 Part A — Theory


### 📖 A1. Reasoning & post-training

1. What is **chain-of-thought (CoT)** prompting, and when does it help?
2. Contrast **instruction tuning (SFT)** with **RL from verifiable rewards (RLVR)**.
3. Why can **pattern matching** succeed on benchmarks without true reasoning?

*Write below:*


#### ✍️ Your answers (A1):

1. **CoT prompting** asks the model to **emit intermediate reasoning steps** before the final answer. It helps on **multi-step math, logic, and planning** where direct answers fail.

2. **SFT** imitates human/teacher demonstrations (behavior cloning). **RLVR** optimizes policies with **scalar rewards from verifiable rules** (e.g., math correctness), enabling exploration beyond static demos.

3. Models can **memorize benchmark patterns** or exploit **spurious correlations** (format hacks, answer leakage) without robust **causal reasoning**.

### 📖 A2. GRPO vs PPO

1. How does GRPO compute the **advantage** $A_i$ without a critic network?
2. What is the role of the **KL penalty** to a reference policy $\pi_{\text{ref}}$?
3. Name one task where rule-based rewards work well and one where they do not.

*Write below:*


#### ✍️ Your answers (A2):

1. GRPO sets $A_i = (r_i - \mu_{\text{group}}) / \sigma_{\text{group}}$ — **within-prompt group normalization** replaces a learned **critic/value network**.

2. The **KL penalty** keeps the optimized policy **close to $\pi_{\text{ref}}$**, preventing **reward hacking** and **language drift** during RL.

3. **Works well:** grade-school math with extractable answers. **Works poorly:** open-ended creative writing or subjective tasks without reliable verifiers.

---

## 💻 Part B — Programming
__Let's implement the core ideas in PyTorch.__


### 🛠️ B1. GRPO group advantage

 GRPO (Group Relative Policy Optimization) replaces a critic network by normalizing rewards **within a group** of completions for the same prompt: $A_i = (r_i - \mu) / \sigma$. This is the core DeepSeek-R1-style advantage used in RL post-training.

In [3]:
def grpo_advantages(rewards: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Normalize rewards within a group: (r - mean) / std."""
    return (rewards - rewards.mean()) / (rewards.std(unbiased=False) + eps)

rewards = torch.tensor([1.0, 0.0, 1.0, 0.5])
adv = grpo_advantages(rewards)
print("Rewards:", rewards.tolist())
print("GRPO advantages:", [round(a, 3) for a in adv.tolist()])

Rewards: [1.0, 0.0, 1.0, 0.5]
GRPO advantages: [0.905, -1.508, 0.905, -0.302]


### 🛠️ B2. Rule-based math verifier

 RL from **verifiable rewards (RLVR)** needs a cheap checker. On GSM8K-style tasks the model must emit a final answer (often after `####`); the verifier returns 1.0 only on an exact match — no human labels required at training time.

In [4]:
def extract_final_answer(text: str):
    """Extract number after '####' marker (GSM8K style) or last integer."""
    m = re.search(r"####\s*(-?\d+(?:\.\d+)?)", text)
    if m:
        return m.group(1)
    nums = re.findall(r"-?\d+(?:\.\d+)?", text)
    return nums[-1] if nums else None

def math_reward(model_output: str, gold: str) -> float:
    pred = extract_final_answer(model_output)
    return 1.0 if pred is not None and str(pred) == str(gold) else 0.0

samples = [("Thought... #### 42", "42"), ("Answer is 41", "42")]
for out, gold in samples:
    print(f"output={out!r} gold={gold} reward={math_reward(out, gold)}")

output='Thought... #### 42' gold=42 reward=1.0
output='Answer is 41' gold=42 reward=0.0


### 🛠️ B3. Reasoning with an open LLM

 **Chain-of-thought (CoT)** prompting asks the model to show intermediate steps before the final answer. Comparing zero-shot vs step-by-step reveals whether the model produces **verifiable reasoning** or jumps directly to an answer.

In [5]:
MODEL = "Qwen/Qwen2.5-Math-1.5B-Instruct"
QUESTION = "If a train travels 60 km in 1.5 hours, what is its average speed in km/h?"

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)
    model.eval()

    def answer(prompt_style: str) -> str:
        prompt = prompt_style + "\n" + QUESTION + "\n"
        inputs = tokenizer(prompt, return_tensors="pt")
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
        return tokenizer.decode(out[0], skip_special_tokens=True)

    zero_shot = answer("Answer:")
    cot = answer("Let's think step by step.")
    print("Zero-shot excerpt:", zero_shot[-200:])
    print("CoT excerpt:", cot[-200:])
except Exception as exc:
    print(f"LLM demo skipped ({exc}). GRPO + verifier implemented above.")
    print("Expected answer: 40 km/h (60 / 1.5).")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/656 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.32k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Zero-shot excerpt: train takes 1.5 hours.
3. Use the formula for average speed: \(\text{Average speed} = \frac{\text{Total distance}}{\text{Total time}}\).

Substitute the given values into the formula:
\[
\text{Average
CoT excerpt: akes 1.5 hours.
3. Use the formula for average speed: \(\text{Average speed} = \frac{\text{Total distance}}{\text{Total time}}\).

Substitute the given values into the formula:
\[
\text{Average speed}


### 🛠️ B4. Self-consistency majority vote

 **Self-consistency** samples multiple CoT completions and takes a **majority vote** on the extracted final answer. It often improves accuracy without changing model weights — a key inference-time reasoning technique from the lecture.

In [6]:
from collections import Counter

def self_consistency_vote(samples: list[str]) -> str:
    answers = [extract_final_answer(s) for s in samples]
    answers = [a for a in answers if a is not None]
    if not answers:
        return None
    return Counter(answers).most_common(1)[0][0]

samples = [
    "Reasoning... #### 40",
    "Step 1... #### 42",
    "Another try... #### 40",
    "Wrong... #### 40",
]
print("Majority answer:", self_consistency_vote(samples))

Majority answer: 40


### 🛠️ B5. GSPO sequence-level advantages

 **GSPO** extends group normalization to the **sequence level**: advantages are computed over entire completions rather than per-token, contrasting with token-level GRPO in B1 when reward is assigned to full answers.

In [7]:
def gspo_advantages(logprobs: torch.Tensor, rewards: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """logprobs: (G, seq_len), rewards: (G,) — normalize at sequence level."""
    seq_logprob = logprobs.sum(dim=-1)
    baseline = (seq_logprob * rewards).sum() / (seq_logprob.sum() + eps)
    return rewards - baseline

logprobs = torch.tensor([[-1.0, -0.5], [-0.8, -0.6], [-1.2, -0.4]])
rewards = torch.tensor([1.0, 0.0, 0.5])
print("GSPO advantages:", gspo_advantages(logprobs, rewards).tolist())

GSPO advantages: [0.4888889193534851, -0.5111110806465149, -0.011111080646514893]


#### 👀 Reflection

With CoT prompting, the model typically shows step-by-step arithmetic. Steps are **partially verifiable** but may still contain **silent errors** — rule-based checks only validate the **final numeric answer**.